In [1]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights


class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0

        self.h = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        Q = self.split(self.W_q(x))
        K = self.split(self.W_k(x))
        V = self.split(self.W_v(x))

        out, weights = scaled_dot_product_attention(Q, K, V, mask)

        B, _, T, _ = out.shape
        out = out.transpose(1, 2).reshape(B, T, self.h * self.d_k)

        return self.W_o(out), weights


Q = torch.randn(2, 5, 8)
K = torch.randn(2, 5, 8)
V = torch.randn(2, 5, 8)

mine, w = scaled_dot_product_attention(Q, K, V)

assert torch.allclose(
    mine,
    F.scaled_dot_product_attention(Q, K, V),
    atol=1e-5
)

assert torch.allclose(w.sum(-1), torch.ones(2, 5))

print("Test 1 passed: Scaled Dot-Product Attention")

causal = torch.tril(torch.ones(5, 5))

mine_c, w_c = scaled_dot_product_attention(Q, K, V, causal)

assert torch.allclose(
    mine_c,
    F.scaled_dot_product_attention(Q, K, V, is_causal=True),
    atol=1e-5
)

print("Test 2 passed: Causal Masking")

x = torch.randn(2, 5, 16)
mha = MultiHeadAttention(16, 4)

ref = nn.MultiheadAttention(16, 4, batch_first=True)

with torch.no_grad():
    for lin, W, b in zip(
        [mha.W_q, mha.W_k, mha.W_v],
        ref.in_proj_weight.chunk(3),
        ref.in_proj_bias.chunk(3)
    ):
        lin.weight.copy_(W)
        lin.bias.copy_(b)

    mha.W_o.weight.copy_(ref.out_proj.weight)
    mha.W_o.bias.copy_(ref.out_proj.bias)

out, weights = mha(x)

assert torch.allclose(out, ref(x, x, x)[0], atol=1e-5)

print("Test 3 passed: Multi-Head Attention")

print("Output shape:", tuple(out.shape))
print("Weights shape:", tuple(weights.shape))

print("\nCausal Attention Weights:")
print(torch.round(w_c[0], decimals=2))

print("\nExperiment 8 completed successfully!")


Test 1 passed: Scaled Dot-Product Attention
Test 2 passed: Causal Masking
Test 3 passed: Multi-Head Attention
Output shape: (2, 5, 16)
Weights shape: (2, 4, 5, 5)

Causal Attention Weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8400, 0.1600, 0.0000, 0.0000, 0.0000],
        [0.2600, 0.1100, 0.6200, 0.0000, 0.0000],
        [0.4400, 0.0600, 0.0600, 0.4400, 0.0000],
        [0.3200, 0.2500, 0.0800, 0.1800, 0.1800]])

Experiment 8 completed successfully!
